In [ ]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 97.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
!pip install transformers torch

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers import DataCollatorForSeq2Seq
import random
from datasets import Dataset
import torch
from collections import defaultdict
import numpy as np
import warnings
import re
from tqdm import tqdm

warnings.filterwarnings(
    "ignore",
    message="Both `max_new_tokens`"
)

In [ ]:
# TODO: Set model size, task, representation, and D here
model_size="t5-base"
task="add" # either "add" or "subtract"
representation="10based" # either "digit", "character", "fixed character", "underscore", "words", "10based", "10ebased"
D = 10 # D to test with
n_samples = 1000
output_dir_name="base-add-10based-D5" # folder that model is in

### **Generating training and development sets with balanced sampling**

In [ ]:
def generate_add_labels(dataset, representation):
    converter = REPRESENTATION_MAP.get(representation, lambda D, n: str(n))
    return [converter(None, num1 + num2) for num1, num2 in dataset]

def generate_subtract_labels(dataset, representation):
    converter = REPRESENTATION_MAP.get(representation, lambda D, n: str(n))
    return [converter(None, num1 - num2) for num1, num2 in dataset]

def format_add_inputs(dataset):
  return [(f"What is {num1} plus {num2}?") for num1, num2 in dataset]

def format_subtract_inputs(dataset):
  return [(f"What is {num1} minus {num2}?") for num1, num2 in dataset]

In [ ]:
def convert_to_character(num):
  '''
  Convert num given in digits to character representation (separated by whitespace)
  Ex: 832 -> 8 3 2
  '''

  str_num = str(num)
  return " ".join(str_num)

def convert_to_fixed_character(D, num):
  '''
  Convert num given in digits to fixed character representation (separated by whitespace)
  Ex: 832 -> 0 8 3 2 if the max is 4 digits
  '''

  str_num = str(num).zfill(D)   # pad with leading zeros
  return ' '.join(str_num)

def convert_to_underscore(num):
  '''
  Convert num given in digits to underscore representation
  Ex: 832 -> 8_3_2
  '''

  str_num = str(num)
  return "_".join(str_num)

def convert_to_words(num):
    '''
    Convert num given in digits to word representation
    Ex: 832 -> eight hundred thirty-two
    '''

    if num == 0:
        return "zero"

    ones = ["", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine",
            "ten", "eleven", "twelve", "thirteen", "fourteen", "fifteen",
            "sixteen", "seventeen", "eighteen", "nineteen"]

    tens = ["", "", "twenty", "thirty", "forty", "fifty",
            "sixty", "seventy", "eighty", "ninety"]

    thousands = ["", "thousand", "million", "billion", "trillion",
                 "quadrillion", "quintillion", "sextillion",
                 "septillion", "octillion", "nonillion", "decillion"]

    def chunk_to_words(n):
        result = ""

        if n >= 100:
            result += ones[n // 100] + " hundred"
            n %= 100
            if n:
                result += " "

        if n >= 20:
            result += tens[n // 10]
            n %= 10
            if n:
                result += "-" + ones[n]
        elif n > 0:
            result += ones[n]

        return result

    words = []
    chunk_index = 0

    while num > 0:
        chunk = num % 1000
        if chunk:
            chunk_words = chunk_to_words(chunk)
            if thousands[chunk_index]:
                chunk_words += " " + thousands[chunk_index]
            words.append(chunk_words)
        num //= 1000
        chunk_index += 1

    return " ".join(reversed(words))

def convert_to_10based(num):
  '''
  Convert num given in digits to 10-based representation
  Ex: 832 -> 8 100 3 10 2
  '''
  if num == 0:
        return "0"


  str_num = str(num)
  length = len(str_num)

  parts = []

  for i, digit in enumerate(str_num):
      if digit != '0':
          power = length - i - 1
          if power == 0:
              parts.append(digit)
          else:
              parts.append(f"{digit} {10**power}")

  return " ".join(parts)

def convert_to_10ebased(num):
  '''
  Convert num given in digits to 10-based representation
  Ex: 832 -> 8 10e2 3 10e1 2 10e0
  '''

  str_num = str(num)
  length = len(str_num)

  parts = []

  for i, digit in enumerate(str_num):
      power = length - i - 1
      parts.append(f"{digit} 10e{power}")

  return " ".join(parts)

REPRESENTATION_MAP = {
  "character": lambda D, n: convert_to_character(n),
  "fixed character": lambda D, n: convert_to_fixed_character(D, n),
  "underscore": lambda D, n: convert_to_underscore(n),
  "words": lambda D, n: convert_to_words(n),
  "10based": lambda D, n: convert_to_10based(n),
  "10ebased": lambda D, n: convert_to_10ebased(n),
}

def apply_representation(data, D, representation):
  converter = REPRESENTATION_MAP.get(representation)
  if converter is None:
      return data

  return [(converter(D, a), converter(D, b)) for a, b in data]

In [ ]:
# D=30
# worst_case = (10**D - 1) + (10**D - 1)
# worst_case_str = convert_to_10based(worst_case)
# tokens = tokenizer(worst_case_str)["input_ids"]
# print(f"Worst case output tokens: {len(tokens)}")

In [ ]:
def generate_train_dataset(D, task, representation, n_samples=1000):

  # Generate n_samples of 2 number inputs
  data = []
  for _ in range(n_samples):
      d = random.randint(2, D) # sample a number from [2, D]

      # randomly sample two numbers from [10**(d-1), 10**d - 1]
      low = 10**(d - 1)
      high = 10**d - 1

      num1 = random.randint(low, high)
      num2 = random.randint(low, high)

      data.append((num1, num2))

  # Generate labels as strings
  labels = []
  if task == "add":
    labels = generate_add_labels(data, representation)
  elif task == "subtract":
    labels = generate_subtract_labels(data, representation)

  # Convert number representation to proper representation if necessary
  data = apply_representation(data, D, representation)

  # Format inputs in form "What is [num1] [operation] [num2]?"
  data_formatted = []

  if task == "add":
    data_formatted = format_add_inputs(data)
  elif task == "subtract":
    data_formatted = format_subtract_inputs(data)

  return data_formatted, labels


In [ ]:
def compute_metrics(eval_pred):
  preds, labels = eval_pred

  if preds.ndim == 3:
      preds = np.argmax(preds, axis=-1)

  preds = np.where(
      (preds < 0) | (preds >= tokenizer.vocab_size),
      tokenizer.pad_token_id,
      preds
  )

  # replace -100 so labels can be decoded
  labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

  decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
  decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

  # EXACT MATCHING
  correct = sum(p.strip().lower() == l.strip().lower() for p, l in zip(decoded_preds, decoded_labels))
  return {"accuracy": correct / len(decoded_labels)}

### **Reload Previously-Saved Model**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Loading model back

model = T5ForConditionalGeneration.from_pretrained("/content/drive/MyDrive/cs4782/Final/models/" + output_dir_name)
tokenizer = T5Tokenizer.from_pretrained("/content/drive/MyDrive/cs4782/Final/models/" + output_dir_name)
model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

### **Computing Model Accuracy**

In [ ]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt")

    # move inputs to same device as model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # use greedy decoding
    outputs = model.generate(
        **inputs,
        max_new_tokens=32,
        num_beams=1, # greedy
        do_sample=False, # no randomness, we want greedy
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

9


In [ ]:
# Generate test sets with random sampling
def generate_test_dataset(D, task, representation, n_samples=1000):

  # Generate n_samples of 2 number inputs
  raw_data = []
  for _ in range(n_samples):
      # randomly sample two numbers with exactly D digits
      low = 10**(D-1)
      high = 10**D - 1

      num1 = random.randint(low, high)
      num2 = random.randint(low, high)

      raw_data.append((num1, num2))

  # Generate labels as strings
  labels = []
  if task == "add":
    labels = generate_add_labels(raw_data, representation)
  elif task == "subtract":
    labels = generate_subtract_labels(raw_data, representation)

  # Convert number representation to proper representation if necessary
  data = apply_representation(raw_data, D, representation)

  # Format inputs in form "What is [num1] [operation] [num2]?"
  data_formatted = []
  if task == "add":
    data_formatted = format_add_inputs(data)
  elif task == "subtract":
    data_formatted = format_subtract_inputs(data)

  return raw_data, data_formatted, labels


In [ ]:
test_data, test_inputs, test_labels = generate_test_dataset(D=D, task=task, representation=representation, n_samples=n_samples)
test_inputs[0:5], test_labels[0:5]

(['What is 2 1000000000 3 100000000 4 10000000 8 1000000 6 100000 6 10000 2 1000 6 100 7 10 6 plus 7 1000000000 5 100000000 3 10000000 5 100000 6 10000 7 1000 3 100 3 10 8?',
  'What is 9 1000000000 5 100000000 1 10000000 1 1000000 6 100000 3 10000 3 1000 4 100 9 10 5 plus 1 1000000000 7 100000000 3 10000000 4 1000000 1 100000 5 10000 7 100 5 10 2?',
  'What is 1 1000000000 1 100000000 6 10000000 7 1000000 9 10000 3 1000 6 10 3 plus 2 1000000000 1 100000000 8 10000000 4 1000000 9 100000 2 10000 1 1000 8 100 3 10 9?',
  'What is 2 1000000000 2 100000000 5 1000000 2 100000 7 10000 4 1000 5 100 2 10 9 plus 5 1000000000 1 100000000 5 10000000 4 1000000 4 100000 7 1000 6 10?',
  'What is 2 1000000000 8 100000000 7 10000000 9 1000000 1 100000 7 10000 2 1000 1 10 8 plus 1 1000000000 2 100000000 1 10000000 9 1000000 7 100000 5 10000 7 1000 1 10 8?'],
 ['9 1000000000 8 100000000 7 10000000 9 1000000 2 100000 3 10000 1 10 4',
  '1 10000000000 1 1000000000 2 100000000 4 10000000 5 1000000 7 10000

### **Evaluate Model**

In [ ]:
def evaluate_model_exact(test_inputs, test_labels):
    correct = 0
    total = len(test_inputs)

    for inp, label in tqdm(zip(test_inputs, test_labels), total=total):
        pred_text = predict(inp)

        # match compute_metrics behavior
        pred = pred_text.strip().lower()
        label = label.strip().lower()
        #print("Prediction:", pred)
        #print("Label:", label)

        if pred == label:
            correct += 1

    return correct / total


acc = evaluate_model_exact(test_inputs, test_labels)
print()
print(f"Accuracy: {acc:.4f}")